In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import mne
from mne_bids import BIDSPath, write_raw_bids
import logging
import matplotlib
matplotlib.use('Qt5Agg')
mne.set_log_level('WARNING')

In [ ]:
# =============================================================================
# Configuration and Constants
# =============================================================================

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

CURDIR = Path.cwd()
DATADIR = Path('/volumes/hyijie_psy/CPP/low_Manning_2021')
DS_ROOT_EEG = DATADIR / 'eegData'

PATH_PREPROCESS_DATA_BIDS = CURDIR.parent / '1_Data'  / 'preprocessData' / 'CPP_low-level-1_Manning_et_al_2021'
PATH_RAW_BIDS = CURDIR.parent / '1_Data'  / 'CPP_low-level-1_Manning_et_al_2021'

os.makedirs(PATH_PREPROCESS_DATA_BIDS, exist_ok=True)
os.makedirs(PATH_RAW_BIDS, exist_ok=True)

FN_FIRST = 'Formatted_'
FN_LAST = '.3_40_preproc'

SAMPLING_RATE = 500

EVENT_ID_MAP = {
    'fixation': 1,
    'boil': 2,
    'photodiode': 3,
    'stimulus': 4,
    'response': 5,
    'offset': 6  
}

SUBJECT_IDS = [f'A{i:03d}' for i in range(1, 21)]

BEHAVIOR_ID_COL = 'subj'
TRIAL_NO_COL = 'trialno'

#EEG_CHANNELS = []
#EOG_CHANNELS = []



In [ ]:
# =============================================================================
# Load and preprocess behavioral data 
# =============================================================================
behavior_raw = pd.read_csv(DS_ROOT_BEH)
behavior_raw = behavior_raw[behavior_raw[BEHAVIOR_ID_COL].astype(str).str.startswith('A')]

all_clean_behavior = []
df_trial_record_all = []
# Process each subject
for sub_id in SUBJECT_IDS:
    logger.info(f"\nProcessing subject: {sub_id}")

    # === Step 1: Filter behavioral data for this subject ===
    beh_sub = behavior_raw[behavior_raw[BEHAVIOR_ID_COL] == sub_id].copy()
    assert not beh_sub.empty, f"No behavioral data found for subject {sub_id}."

    # Convert trial numbers from 1-based to 0-based
    beh_sub[TRIAL_NO_COL] = beh_sub[TRIAL_NO_COL] - 1

    # === Step 2: Load EEG .mat file ===
    eeg_path = os.path.join(DS_ROOT_EEG, FN_FIRST + sub_id + FN_LAST + '.mat')
    assert os.path.exists(eeg_path), f"EEG .mat file not found: {eeg_path}"

    # extract eeg data, good trials (which is a derivation of preprocess analysis) ,triggers and trigger idx (which is the start and end trigger for each trial)
    data_mat = scipy.io.loadmat(eeg_path)
    data_EEG = data_mat['X']
    index_good_trials = data_mat['GoodTrial'][0][0][0].T
    triggers = data_mat['Onsets']['corrMatrix'][0, 0]
    trial_epoch_idx = data_mat['trialEpochIdx']
    count_trial = len(triggers)

    # Build dataframes
    good_trials = pd.DataFrame(index_good_trials, columns=['is_good'])
    triggers_df = pd.DataFrame(triggers)
    trial_epoch_idx_df = pd.DataFrame(trial_epoch_idx, columns=['start_idx', 'end_idx'])

    # Add an index column each dataFrame, which is used for alignment and index
    for df in [good_trials, triggers_df, trial_epoch_idx_df]:
        df[TRIAL_NO_COL] = np.arange(len(df))

    # === Step 3: Exclude trials with missing response triggers ===
    valid_resp = ~triggers_df.iloc[:, 4].isna()
    valid_trial_indices = np.where(valid_resp)[0]
    count_na = triggers_df.iloc[:, 4].isna().sum()
    beh_trial_indices = beh_sub[TRIAL_NO_COL].values

    # Validate alignment between behavior data and eeg data
    assert np.array_equal(valid_trial_indices, beh_trial_indices), \
        f"Subject {sub_id}: trial indices mismatch between EEG triggers and behavior."
    
    logger.info(f"Subject {sub_id}: behavioral and EEG trial indices aligned.")

    # Filter to valid-response trials
    triggers_valid = triggers_df[triggers_df[TRIAL_NO_COL].isin(valid_trial_indices)]
    trial_epoch_idx_valid = trial_epoch_idx_df[trial_epoch_idx_df[TRIAL_NO_COL].isin(valid_trial_indices)]
    good_trials_valid = good_trials[good_trials[TRIAL_NO_COL].isin(valid_trial_indices)]
    beh_valid = beh_sub[beh_sub[TRIAL_NO_COL].isin(valid_trial_indices)]

    # === Step 4: Keep only "good" trials ===
    good_mask = good_trials_valid['is_good'] == 1
    good_trial_no = good_trials_valid.loc[good_mask, TRIAL_NO_COL]

    # Filter to good trials
    triggers_good = triggers_valid[triggers_valid[TRIAL_NO_COL].isin(good_trial_no)]
    trial_epoch_idx_good = trial_epoch_idx_valid[trial_epoch_idx_valid[TRIAL_NO_COL].isin(good_trial_no)]
    beh_good = beh_valid[beh_valid[TRIAL_NO_COL].isin(good_trial_no)]
    count_bad_trial = len(triggers_valid) - len(triggers_good)

    # Ensure alignment preserved
    assert np.array_equal(triggers_good[TRIAL_NO_COL].values, beh_good[TRIAL_NO_COL].values), \
        f"Post-good-trial alignment failed for {sub_id}"

    # === Step 5: Filter coherence trials ===
    beh_filter_condition = beh_good[(beh_good['coherence'] != 1) & (beh_good['coherence'] != 0.1)]
    count_filter_condition = len(triggers_good) - len(beh_filter_condition)
    filter_no = beh_filter_condition[TRIAL_NO_COL].tolist()
    triggers_filter_condition = triggers_good[triggers_good[TRIAL_NO_COL].isin(filter_no)]
    
    # === Step 6: Remove RT outliers ===
    rt_mean = np.nanmean(beh_filter_condition['RT'])
    rt_std = np.nanstd(beh_filter_condition['RT'])
    rt_lower = max(0.2, rt_mean - 3 * rt_std)
    rt_upper = rt_mean + 3 * rt_std

    beh_clean = beh_filter_condition[
        (beh_filter_condition['RT'] >= rt_lower) &
        (beh_filter_condition['RT'] <= rt_upper)
    ]
    count_clean = len(beh_clean)

    count_outlier_rt = len(beh_filter_condition) - len(beh_clean)
    if beh_clean.empty:
        logger.warning(f"Subject {sub_id}: no valid trials after RT filtering. Skipping.")
        continue

    clean_trial_no = beh_clean[TRIAL_NO_COL]
    triggers_clean = triggers_filter_condition[triggers_filter_condition[TRIAL_NO_COL].isin(clean_trial_no)]
    trial_epoch_idx_clean = trial_epoch_idx_good[trial_epoch_idx_good[TRIAL_NO_COL].isin(clean_trial_no)]

    # === Step 7: Prepare continuous EEG data for BIDS ===
    eeg_continuous = np.concatenate([trial for trial in data_EEG[0]], axis=1)/1e6
    montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
    ch_names = montage.ch_names
    assert eeg_continuous.shape[0] == len(ch_names), \
        f"Channel count mismatch for {sub_id}: EEG has {eeg_continuous.shape[0]} channels, " \
        f"but {len(ch_names)} names provided."

    info = mne.create_info(ch_names=ch_names, sfreq=SAMPLING_RATE, ch_types='eeg')
    data_eeg_bids = mne.io.RawArray(eeg_continuous, info)

    # === Step 8: Build BIDS event array ===
    offsets = trial_epoch_idx_clean['start_idx'].values - 1  # MATLAB 1-based → Python 0-based
    triggers_continuous = triggers_clean.copy()
    numeric_cols = [col for col in triggers_continuous.columns if col != TRIAL_NO_COL]
    triggers_continuous[numeric_cols] = triggers_continuous[numeric_cols].values + offsets[:, None]

    onsets = []
    unused = []
    event_ids = []

    for _, row in triggers_continuous.iterrows():
        trial_onsets = row[numeric_cols].values - 1  # to 0-based sample index
        n_events = len(trial_onsets)
        onsets.extend(trial_onsets)
        unused.extend([0] * n_events)
        event_ids.extend(range(1, n_events + 1))

    events = np.column_stack([onsets, unused, event_ids]).astype(int)

    # === Step 9: Write BIDS dataset ===

    bids_path = BIDSPath(
        subject=sub_id,
        task='randomDot',
        datatype='eeg',
        root=PATH_PREPROCESS_DATA_BIDS
    )
    
    write_raw_bids(
        raw=data_eeg_bids,
        bids_path=bids_path,
        events=events,
        event_id=EVENT_ID_MAP,
        format='BrainVision',
        allow_preload=True,
        overwrite=True,
        verbose=False
    )

    # === Step 9: Save cleaned behavioral data in BIDS format ===
    # Construct the behavioral file paths
    bids_path_beh = os.path.join(PATH_PREPROCESS_DATA_BIDS, 'sub-'+sub_id, 'beh')
    os.makedirs(bids_path_beh, exist_ok=True)
    beh_save_path = os.path.join(bids_path_beh, f"sub-{sub_id}_task-randomDot_beh.csv")
    beh_clean.to_csv(beh_save_path, sep=',', index=False, na_rep='n/a')

    logger.info(f"Subject {sub_id}: BIDS conversion completed.")
    all_clean_behavior.append(beh_clean)    

    # === Step 10: Record trial ==
    df_trial_record = {
        'sub_id': sub_id,
        'count_trial': count_trial,
        'count_na': count_na,
        'count_bad_trial': count_bad_trial,
        'count_filter_condition': count_filter_condition,
        'count_outlier_rt': count_outlier_rt,
        'count_clean': count_clean,
        'percent_clean': count_clean / count_trial
        }
    df_trial_record_all.append(df_trial_record)

# =============================================================================
# Final aggregation
# =============================================================================
if all_clean_behavior:
    final_df = pd.concat(all_clean_behavior, ignore_index=True)
    selected = final_df[['subj', 'coherence', 'RT', 'accuracy', 'keypress']]
    selected.to_csv(os.path.join(PATH_PREPROCESS_DATA_BIDS, 'behavioral_full.csv'), index=False)
    logger.info(f"Final behavioral data saved to {os.path.join(PATH_PREPROCESS_DATA_BIDS, 'behavioral_full.csv')}")

else:
    logger.warning("No subjects processed successfully.")

if df_trial_record_all:
    df_trial_record_all = pd.DataFrame(df_trial_record_all)
    df_trial_record_all.to_csv(os.path.join(PATH_PREPROCESS_DATA_BIDS, 'trial_record.csv'), index=False)
